# Raccolta Dati da Reddit — r/Italia, keyword: *notizie, film, sport*
Raccolta tramite **Arctic Shift** (archivio pubblico Reddit per ricerca accademica) di:
- **100 post per keyword** (300 totali) di r/Italia
- e i **top 50 commenti** per ciascun post.

Le parole cercate in ogni post sono: **notizie, film, sport**.

## Installazione dipendenze

In [1]:
# !pip install requests spacy tqdm
# !python -m spacy download it_core_news_sm

## Configurazione

In [17]:
import requests
import time
import calendar
import pandas as pd
from datetime import datetime

SUBREDDIT         = "Italia"
KEYWORDS          = ["notizie", "film", "sport"]
N_POSTS_PER_KW    = 100
COMMENTS_PER_POST = 50

BASE_URL = "https://arctic-shift.photon-reddit.com/api"
HEADERS  = {"User-Agent": "python:elita.tesi.multi:v1.0 (academic NLP)"}

print("Configurazione:")
print(f"  Subreddit             : r/{SUBREDDIT}")
print(f"  Keywords              : {KEYWORDS}")
print(f"  Post per keyword      : {N_POSTS_PER_KW}  (totale: {N_POSTS_PER_KW * len(KEYWORDS)})")
print(f"  Commenti per post     : {COMMENTS_PER_POST}")

Configurazione:
  Subreddit             : r/Italia
  Keywords              : ['notizie', 'film', 'sport']
  Post per keyword      : 100  (totale: 300)
  Commenti per post     : 50


## Test connessione API
Attenzione, con servizi intermedi come **Arctic Shift** le risposte possono variare. Se lo status è 200 OK, altrimenti se è 422 riprova dopo qualche secondo.

In [3]:
# Test 1: endpoint posts/search (prima keyword)
resp = requests.get(f"{BASE_URL}/posts/search", headers=HEADERS, timeout=20, params={
    "subreddit": SUBREDDIT,
    "title": KEYWORDS[0],
    "limit": 2,             # risultati restituiti per chiamata
    "after":  "2025-01-01",
    "before": "2025-01-31"
})
print(f"[posts/search]    Status: {resp.status_code}")
if resp.status_code == 200 and resp.json().get("data"):
    p = resp.json()["data"][0]
    sample_post_id = p.get("id", "")
    print(f"  Titolo   : {p.get('title','')[:80]}")
    print(f"  Score    : {p.get('score',0)}  |  Commenti: {p.get('num_comments',0)}")
    print(f"  Post ID  : {sample_post_id}")
else:
    sample_post_id = ""
    print(f"  Errore: {resp.text[:200]}")

# Test 2: endpoint comments/search per un post specifico
if sample_post_id:
    resp2 = requests.get(f"{BASE_URL}/comments/search", headers=HEADERS, timeout=20, params={
        "link_id": sample_post_id,
        "limit": 3
    })
    print(f"\n[comments/search] Status: {resp2.status_code}")
    if resp2.status_code == 200 and resp2.json().get("data"):
        print(f"  Primo commento: {resp2.json()['data'][0].get('body','')[:150]}")
    else:
        print(f"  Errore: {resp2.text[:200]}")

[posts/search]    Status: 200
  Titolo   : Il Pil dell'Italia è fermo, la disoccupazione sale al 6,2% - Notizie
  Score    : 241  |  Commenti: 120
  Post ID  : 1idmjsb

[comments/search] Status: 200
  Primo commento: [removed]


## Raccolta Post

In r/Italia si vogliono ottenere i post con keyword = [notizie, film, sport] nel **titolo** tramite `/api/posts/search`.

Finestre mensili per evitare il rate limit (stessa tecnica dei commenti).

In [4]:
def collect_posts(subreddit, keyword, target=100):
    all_posts = []

    time_windows = []
    for year in [2025, 2024, 2023]:
        max_month = 5 if year == 2025 else 12
        for month in range(max_month, 0, -1):
            last_day = calendar.monthrange(year, month)[1]
            time_windows.append((f"{year}-{month:02d}-01", f"{year}-{month:02d}-{last_day}", f"{year}-{month:02d}"))

    for after, before, label in time_windows:
        if len(all_posts) >= target:
            break

        params = {"subreddit": subreddit, "title": keyword, "limit": 25, "after": after, "before": before}
        last_utc = None

        while len(all_posts) < target:
            if last_utc:
                params["before"] = last_utc
            try:
                resp = requests.get(f"{BASE_URL}/posts/search", headers=HEADERS, params=params, timeout=30)
                if resp.status_code == 422:
                    break
                resp.raise_for_status()
                items = resp.json().get("data", [])
            except Exception as e:
                print(f"  Errore {label}: {e}")
                break

            if not items:
                break

            for p in items:
                if len(all_posts) >= target:
                    break
                title = p.get("title", "").strip()
                if not title:
                    continue
                selftext = p.get("selftext", "").strip()
                if selftext in ("[removed]", "[deleted]"):
                    selftext = ""
                ts = float(p.get("created_utc", 0))
                all_posts.append({
                    "post_id"     : p.get("id", ""),
                    "title"       : title,
                    "selftext"    : selftext,
                    "author"      : p.get("author", ""),
                    "timestamp"   : datetime.utcfromtimestamp(ts).isoformat(),
                    "score"       : p.get("score", 0),
                    "num_comments": p.get("num_comments", 0),
                    "url"         : "https://reddit.com" + p.get("permalink", "") if p.get("permalink") else "",
                })

            oldest_utc = min(float(p.get("created_utc", 0)) for p in items)
            last_utc   = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")
            if len(items) < 50:
                break
            time.sleep(1)

        print(f"  [{keyword}] {label}: {len(all_posts)} post totali", end="\r")
        time.sleep(1.5)

    print(f"\n  [{keyword}] Post raccolti: {len(all_posts)}")
    return all_posts

# Raccolta per tutte le keyword
all_posts = []
for kw in KEYWORDS:
    print(f"\n--- Raccolta keyword: '{kw}' ---")
    posts_kw = collect_posts(SUBREDDIT, kw, target=N_POSTS_PER_KW)
    for p in posts_kw:
        p["keyword"] = kw
    all_posts.extend(posts_kw)

df_posts = pd.DataFrame(all_posts)
print(f"\nPost totali raccolti: {len(df_posts)}")
print(df_posts.groupby("keyword").size().rename("n_post"))


--- Raccolta keyword: 'notizie' ---
  [notizie] 2024-06: 100 post totali
  [notizie] Post raccolti: 100

--- Raccolta keyword: 'film' ---
  [film] 2024-12: 100 post totali
  [film] Post raccolti: 100

--- Raccolta keyword: 'sport' ---
  [sport] 2023-01: 84 post totali
  [sport] Post raccolti: 84

Post totali raccolti: 284
keyword
film       100
notizie    100
sport       84
Name: n_post, dtype: int64


## Raccolta Commenti per Post
Per ogni post, si vogliono raccoglire fino a **100 commenti** dall'API (ordinati per `created_utc` discendente), poi li si ordinerà per score decrescente tenedo i **top 50**.

Questo garantisce che i commenti selezionati siano quelli più apprezzati dalla community e quindi che rappresentano meglio la reazione collettiva al post.

In [6]:
### ATTENZIONE non eseguire se non vuoi perdere 10 min di vita !!!!!!!!!!!!!!
def collect_comments_for_posts(df_posts, n_comments=50, fetch_limit=100):
    """
    Per ogni post:
      - recupera fino a `fetch_limit` commenti dall'API paginando correttamente
      - ordina per score decrescente
      - tiene i top `n_comments`
    """
    all_comments = []

    for i, row in df_posts.iterrows():
        post_id  = row["post_id"]
        raw_items = []
        last_utc  = None

        # Paginazione: continua finché non raggiungiamo fetch_limit o l'API non ha più risultati
        while len(raw_items) < fetch_limit:
            params = {"link_id": post_id, "limit": 100}
            if last_utc:
                params["before"] = last_utc

            try:
                resp = requests.get(f"{BASE_URL}/comments/search", headers=HEADERS,
                                    params=params, timeout=30)
                if resp.status_code != 200:
                    break
                items = resp.json().get("data", [])
            except Exception as e:
                print(f"  Errore post {post_id}: {e}")
                break

            if not items:
                break

            raw_items.extend(items)

            # Aggiorna il cursore PRIMA di decidere se continuare
            oldest_utc = min(float(c.get("created_utc", 0)) for c in items)
            last_utc   = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")

            if len(items) < 100:    # Se l'API ha restituito meno di 100, non ci sono altre pagine
                break

            time.sleep(0.3)

        # Filtra rimossi/troppo corti, ordina per score, tieni i top n
        valid = [
            c for c in raw_items
            if c.get("body", "").strip() not in ("", "[deleted]", "[removed]")
            and len(c.get("body", "").strip()) >= 10
        ]
        top = sorted(valid, key=lambda c: c.get("score", 0), reverse=True)[:n_comments]

        for rank, c in enumerate(top, start=1):
            ts = float(c.get("created_utc", 0))
            all_comments.append({
                "comment_id"   : c.get("id", ""),
                "post_id"      : post_id,
                "rank_by_score": rank,
                "comment_text" : c.get("body", "").strip(),
                "author"       : c.get("author", ""),
                "timestamp"    : datetime.utcfromtimestamp(ts).isoformat(),
                "score"        : c.get("score", 0),
                "permalink"    : "https://reddit.com" + c.get("permalink", "")
                                  if c.get("permalink") else "",
            })

        print(f"  [{i+1:3d}/{len(df_posts)}] {post_id}: "
              f"{len(raw_items)} recuperati → {len(valid)} validi → top {len(top)}")
        time.sleep(0.8)

    print(f"\nCommenti totali: {len(all_comments)}")
    return all_comments

comments = collect_comments_for_posts(df_posts, n_comments=COMMENTS_PER_POST, fetch_limit=100)
df_comments = pd.DataFrame(comments)
print(df_comments[["comment_id", "post_id", "rank_by_score", "score", "comment_text"]].head(8))

  [  1/284] 1kwlwc6: 26 recuperati → 25 validi → top 25
  [  2/284] 1kuc88y: 1 recuperati → 1 validi → top 1
  [  3/284] 1kq6axq: 33 recuperati → 31 validi → top 31
  [  4/284] 1kna8hg: 10 recuperati → 10 validi → top 10
  [  5/284] 1kn9cp6: 59 recuperati → 58 validi → top 50
  [  6/284] 1kjv1dn: 1 recuperati → 1 validi → top 1
  [  7/284] 1khu4t1: 0 recuperati → 0 validi → top 0
  [  8/284] 1khu16n: 0 recuperati → 0 validi → top 0
  [  9/284] 1khu0wh: 0 recuperati → 0 validi → top 0
  [ 10/284] 1khu0sz: 0 recuperati → 0 validi → top 0
  [ 11/284] 1kem3wd: 1 recuperati → 1 validi → top 1
  [ 12/284] 1ke1jvk: 1 recuperati → 1 validi → top 1
  [ 13/284] 1kd66x8: 5 recuperati → 4 validi → top 4
  [ 14/284] 1kczbl2: 5 recuperati → 5 validi → top 5
  [ 15/284] 1kcz8yh: 56 recuperati → 54 validi → top 50
  [ 16/284] 1kc8tre: 6 recuperati → 6 validi → top 6
  [ 17/284] 1ka1u9n: 3 recuperati → 3 validi → top 3
  [ 18/284] 1k9s902: 1 recuperati → 1 validi → top 1
  [ 19/284] 1k7hecq: 3 recupera

## Salvataggio CSV
- `posts_Italia_multi.csv` — un post per riga, con colonna `keyword`
- `comments_Italia_multi.csv` — un commento per riga, con `post_id` come chiave di join
- `corpus_Italia_multi.csv` — long format: una riga per documento (post o commento), con `keyword` per filtrare per topic

In [7]:
df_posts.to_csv("posts_Italia_multi.csv", index=False, encoding="utf-8-sig")
print(f"Post salvati    : {len(df_posts)} righe → 'posts_Italia_multi.csv'")

df_comments.to_csv("comments_Italia_multi.csv", index=False, encoding="utf-8-sig")
print(f"Commenti salvati: {len(df_comments)} righe → 'comments_Italia_multi.csv'")

# Corpus long-format con colonna keyword
posts_long = df_posts.assign(
    type="post",
    doc_id=df_posts["post_id"],
    text=df_posts.apply(
        lambda r: (r["title"] + "\n" + r["selftext"]).strip() if r["selftext"] else r["title"],
        axis=1
    ),
    rank_by_score=None,
)[["doc_id", "post_id", "keyword", "type", "text", "author", "timestamp", "score", "rank_by_score"]]

comments_long = df_comments.merge(df_posts[["post_id", "keyword"]], on="post_id", how="left") \
    .rename(columns={"comment_id": "doc_id", "comment_text": "text"}) \
    .assign(type="comment")[["doc_id", "post_id", "keyword", "type", "text", "author", "timestamp", "score", "rank_by_score"]]

df_corpus = pd.concat([posts_long, comments_long], ignore_index=True)
df_corpus.to_csv("corpus_Italia_multi.csv", index=False, encoding="utf-8-sig")
print(f"Corpus salvato  : {len(df_corpus)} righe → 'corpus_Italia_multi.csv'")
print()
print(df_corpus.groupby(["keyword", "type"]).size().unstack(fill_value=0))

Post salvati    : 284 righe → 'posts_Italia_multi.csv'
Commenti salvati: 4027 righe → 'comments_Italia_multi.csv'
Corpus salvato  : 4311 righe → 'corpus_Italia_multi.csv'

type     comment  post
keyword               
film        1196   100
notizie     1743   100
sport       1088    84


## Statistiche del corpus

In [8]:
import plotly.express as px

print("=" * 50)
print("STATISTICHE CORPUS")
print("=" * 50)
print(f"Post totali              : {len(df_posts)}")
print(f"  - con testo (selftext) : {(df_posts['selftext'] != '').sum()}")
print(f"  - solo link            : {(df_posts['selftext'] == '').sum()}")
print(f"Commenti totali          : {len(df_comments)}")
print(f"Media commenti per post  : {len(df_comments)/len(df_posts):.1f}")

# Post per mese
df_posts["mese"] = df_posts["timestamp"].str[:7]
fig1 = px.bar(df_posts["mese"].value_counts().sort_index().reset_index(),
              x="mese", y="count", title="Post per mese",
              labels={"mese": "Mese", "count": "Post"})
fig1.show()

# Distribuzione commenti per post
commenti_per_post = df_comments.groupby("post_id").size().reset_index(name="n_commenti")
fig2 = px.histogram(commenti_per_post, x="n_commenti", nbins=26,
                    title="Distribuzione commenti raccolti per post",
                    labels={"n_commenti": "Commenti raccolti", "count": "Post"})
fig2.show()

# Top 10 post per score
print("\nTop 10 post per score:")
display(df_posts.nlargest(10, "score")[["title", "score", "num_comments", "timestamp"]])

STATISTICHE CORPUS
Post totali              : 284
  - con testo (selftext) : 139
  - solo link            : 145
Commenti totali          : 4027
Media commenti per post  : 14.2



Top 10 post per score:


,title,score,num_comments,timestamp
113,"""Piratare"" i film e serie tv è l'unico modo ch...",613,239,2025-05-20T14:17:24
144,27 MARZO 1975 🇮🇹 Nelle sale cinematografiche d...,599,56,2025-03-27T07:47:16
179,Perché ogni Natale si guarda sempre Una poltro...,496,247,2024-12-24T19:33:05
110,"I 10 film con maggiori incassi in Italia, che ...",482,151,2025-05-21T19:31:57
108,"Se questo momento storico fosse un film, il ge...",390,125,2025-05-22T11:21:58
147,"Domanda seria, perché i registi italiani non f...",270,147,2025-03-23T18:31:55
14,Notizie top tier,262,54,2025-05-02T12:03:46
42,"Il Pil dell'Italia è fermo, la disoccupazione ...",241,120,2025-01-30T13:07:36
23,Le grandi notizie di Repubblica,211,25,2025-03-16T23:13:06
53,Studenti italiani campioni europei dei compiti...,171,50,2024-12-06T22:04:26


## Tokenizzazione, Lemmatizzazione e POS-tagging

Verrà utilizzato il modello italiano `it_core_news_sm` di spaCy per analizzare tutti i documenti del corpus (post e commenti). Per ogni token estraiamo:
- `token`: testo originale
- `lemma`: forma base in minuscolo
- `pos`: parte del discorso (NOUN, VERB, ADJ, EMOJI, ecc.)

Filtriamo spazi e punteggiatura. L'identificatore è `doc_id` (che vale sia per post che commenti).

Salviamo il risultato in `tokens_Italia_notizie.csv`.

In [11]:
import spacy
import emoji as emoji_lib
from tqdm.auto import tqdm

nlp = spacy.load("it_core_news_sm")

def process_text(text):
    doc = nlp(str(text))
    tokens = []
    for token in doc:
        if token.is_space or token.is_punct:
            continue
        pos = "EMOJI" if emoji_lib.emoji_count(token.text) > 0 else token.pos_
        tokens.append({
            "token" : token.text,
            "lemma" : token.lemma_.lower(),
            "pos"   : pos,
        })
    return tokens

print(f"Processamento di {len(df_corpus)} documenti con spaCy...")

all_tokens = []
for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus)):
    for t in process_text(row["text"]):
        all_tokens.append({"doc_id": row["doc_id"], "keyword": row["keyword"], **t})

df_tokens = pd.DataFrame(all_tokens)

print(df_tokens.groupby("keyword").size().rename("n_token"))

df_tokens.to_csv("tokens_Italia_multi.csv", index=False, encoding="utf-8-sig")
print(f"\nToken salvati in 'tokens_Italia_multi.csv'")

Processamento di 4311 documenti con spaCy...


100%|██████████| 4311/4311 [00:33<00:00, 129.73it/s]


keyword
film       38318
notizie    61345
sport      46523
Name: n_token, dtype: int64

Token salvati in 'tokens_Italia_multi.csv'


In [16]:
print(df_tokens['pos'].value_counts().to_string())

pos
NOUN     29487
ADP      20016
VERB     19639
DET      15450
ADV      13240
PRON     10548
ADJ       9468
AUX       8566
PROPN     6978
CCONJ     5695
SCONJ     3819
NUM       2028
X          568
INTJ       307
EMOJI      177
PUNCT       95
SYM         87
PART        18


NOUN = nomi, ADJ = aggettivi, VERB = verbi, AUX = ausiliari, EMOJI = emoji.
ADP = preposizioni, CCONJ = congiunzioni coordinanti, SCONJ = congiunzioni subordinanti, DET = determinatori, PRON = pronomi, ADV = avverbi, PROPN = nomi propri, NUM = numeri, PART = particelle, INTJ = interiezioni, PUNCT = punteggiatura.